In [2]:
print("hello")

hello


In [3]:
!pip install mysql-connector-python

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
!pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from dotenv import load_dotenv
import os

load_dotenv()

hostname = os.getenv("hostname")
username = os.getenv("user")
password = os.getenv("password")
database = os.getenv("database")
port = os.getenv("port")

In [19]:
import mysql.connector
from mysql.connector import Error

connection = None

try:
    connection = mysql.connector.connect(
        host=hostname,
        database=database,
        user=username,
        password=password,
        port=port
    )

    if connection.is_connected():
        db_Info = connection.get_server_info()
        print("Connected to MySQL Server version", db_Info)

        cursor = connection.cursor()
        cursor.execute("SELECT DATABASE();")

        record = cursor.fetchone()
        print("You're connected to database:", record)

except Error as e:
    print("Error while connecting to MySQL:", e)

finally:
    if connection and connection.is_connected():
        cursor.close()
        connection.close()
        print("MySQL connection is closed")


Error while connecting to MySQL: 2005 (HY000): Unknown MySQL server host '6bnvj.h.filess.io' (11001)


In [24]:
import pandas as pd
order_payments = pd.read_csv("data/raw/olist_order_payments_dataset.csv")
order_payments.head()
order_payments.shape

(103886, 5)

In [26]:
import pandas as pd
import mysql.connector
from mysql.connector import Error

# CSV file path
csv_file_path = "data/raw/olist_order_payments_dataset.csv"

# Table name
table_name = "olist_order_payments"

connection = None
cursor = None

try:
    # Step 1: Establish connection
    connection = mysql.connector.connect(
        host=hostname,
        database=database,
        user=username,
        password=password,
        port=port
    )

    if connection.is_connected():
        print("Connected to MySQL Server successfully!")

        # Step 2: Create cursor
        cursor = connection.cursor()

        # Step 3: Drop table if exists
        cursor.execute(f"DROP TABLE IF EXISTS {table_name}")
        print(f"Table `{table_name}` dropped if it existed.")

        # Step 4: Create table
        create_table_query = f"""
        CREATE TABLE {table_name} (
            order_id VARCHAR(50),
            payment_sequential INT,
            payment_type VARCHAR(20),
            payment_installments INT,
            payment_value FLOAT
        )
        """
        cursor.execute(create_table_query)
        print(f"Table `{table_name}` created successfully!")

        # Step 5: Load CSV
        data = pd.read_csv(csv_file_path)
        print("CSV data loaded into pandas DataFrame.")

        # Step 6: Batch insert
        batch_size = 500
        total_records = len(data)

        insert_query = f"""
        INSERT INTO {table_name}
        (order_id, payment_sequential, payment_type, payment_installments, payment_value)
        VALUES (%s, %s, %s, %s, %s)
        """

        print(f"Starting data insertion in batches of {batch_size}...")

        for start in range(0, total_records, batch_size):
            end = start + batch_size
            batch = data.iloc[start:end]

            batch_records = [
                tuple(row) for row in batch.itertuples(index=False, name=None)
            ]

            cursor.executemany(insert_query, batch_records)
            connection.commit()

            print(f"Inserted records {start + 1} to {min(end, total_records)}")

        print(f"All {total_records} records inserted successfully.")

except Error as e:
    print("Error while connecting to MySQL or inserting data:", e)

finally:
    if connection and connection.is_connected():
        cursor.close()
        connection.close()
        print("MySQL connection is closed.")

Error while connecting to MySQL or inserting data: 2005 (HY000): Unknown MySQL server host '6bnvj.h.filess.io' (11001)


In [ ]:
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.6/313.6 kB 20.2 MB/s eta 0:00:00


In [32]:
from dotenv import load_dotenv
import os

load_dotenv()

hostname = os.getenv("mongodb_hostname")
username = os.getenv("mongodb_username")
password = os.getenv("mongodb_password")
database = os.getenv("mongodb_database")
port = int(os.getenv("mongodb_port"))

In [34]:
# importing module
from pymongo import MongoClient



uri = "mongodb://" + username + ":" + password + "@" + hostname + ":" + str(port) + "/" + database

# Connect with the portnumber and host
client = MongoClient(uri)

# Access database
mydatabase = client[database]


In [35]:
# prompt: read the product_category csv and create a collection and upload it to above mongoDB

import pandas as pd
from pymongo import MongoClient

# Load the product_category CSV file into a pandas DataFrame
try:
  product_category_df = pd.read_csv("product_category_name_translation.csv")
except FileNotFoundError:
  print("Error: 'product_category_name_translation.csv' not found.")
  exit() # Exit the script if the file is not found


# MongoDB connection details (assuming these are already defined in your script)
hostname = "p8io3.h.filess.io"
database = "olistDataNoSQL_becomingit"
port = "27018"
username = "olistDataNoSQL_becomingit"
password = "dcea9f94c68907e3b1f73fcbba26645ef83470e3"

uri = "mongodb://" + username + ":" + password + "@" + hostname + ":" + port + "/" + database

try:
    # Establish a connection to MongoDB
    client = MongoClient(uri)
    db = client[database]

    # Select the collection (or create if it doesn't exist)
    collection = db["product_categories"]  # Choose a suitable name for your collection

    # Convert the DataFrame to a list of dictionaries for insertion into MongoDB
    data_to_insert = product_category_df.to_dict(orient="records")

    # Insert the data into the collection
    collection.insert_many(data_to_insert)

    print("Data uploaded to MongoDB successfully!")

except Exception as e:
    print(f"An error occurred: {e}")

finally:
    # Close the MongoDB connection
    if client:
        client.close()

Error: 'product_category_name_translation.csv' not found.
An error occurred: name 'product_category_df' is not defined
